# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmerSajid842/flyrankmlproject/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Starter data not found in the expected repo locations.")

df = pd.read_csv(data_path)
df.head()


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
df[["ctr", "avg_position", "engagement_rate", "scroll_rate", "trend_direction", "trend_pct"]].describe().T


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
feature_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "content_age_days", "word_count"]
y_train = train_df["trend_direction"].eq("down")
y_test = test_df["trend_direction"].eq("down")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(train_df[feature_cols].fillna(0), y_train)
scores = model.predict_proba(test_df[feature_cols].fillna(0))[:, 1]
roc_auc_score(y_test, scores)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
from sklearn.metrics import precision_score, recall_score
preds = (scores >= 0.5).astype(int)
precision_score(y_test, preds), recall_score(y_test, preds)


## 5. Limitations

*What this work cannot claim.*

In [ ]:
df.groupby("trend_direction")["ctr"].mean().to_frame(name="mean_ctr")


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
result_table = pd.DataFrame({
    "question": ["Which pages should the SEO team refresh first?"],
    "method": ["client-holdout random forest on measurable engagement and position features"],
    "claim": ["Directional decision-support under a held-out client split"],
})
result_table


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
print("This notebook is ready for the final capstone narrative and artifact exports.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.